# 06 — MHW-metric composites

Per-ENSO-year fields (Jun(0)–May(+1)) are composited by phase. Composite = mean over
phase years − mean over all ENSO years {ENSO_YEARS}, after removing a linear trend across years if
`cfg.COMPOSITE_DETREND`. Significance: resampling null (random draws of the same number of years,
`cfg.N_BOOT`), two-sided; maps stippled where significant after Benjamini–Hochberg FDR
(`ALPHA_FDR`, Wilks 2016). Regional curves: filled markers = pointwise p < 0.05.
With n≈3–4 strong El Niños the null is coarse — treat individual-event lines as part of the result.

Seasonal metrics per ENSO year: MHW days, cat ≥2 days, cumulative intensity (sums over the season),
mean intensity on MHW days (cum_int / mhw_days; NaN where no MHW days), max intensity.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
table = cfg.load_enso_table()
groups = cfg.phase_years(table)
masks = xr.open_dataset(cfg.F_MASKS)
names = masks.attrs["region_names"].split(";")
comps = ["El Niño", "Strong El Niño", "La Niña"]
mm = xr.open_dataset(cfg.F_MHW_MONTHLY)
ym = cfg.to_enso_year_month(mm)

In [ ]:
seas = xr.Dataset({v: cfg.to_seasons(ym[v], "sum") for v in ["mhw_days", "cat2_days", "cum_int"]})
seas["max_int"] = cfg.to_seasons(ym.max_int.fillna(-np.inf), "max").where(lambda x: np.isfinite(x))
seas["mean_int"] = (seas.cum_int / seas.mhw_days).where(seas.mhw_days > 0)
seas = seas.where(masks.ocean)

In [ ]:
PLOT = {"mhw_days": ("MHW days per season", 30),
        "cat2_days": ("Cat ≥2 MHW days per season", 15),
        "mean_int": ("Mean MHW intensity (°C)", 1.0)}
res = {}
for v, (lab, vmax) in PLOT.items():
    res[v] = {k: cfg.composite_bootstrap(seas[v], groups[k]) for k in comps}
    fig = cfg.plot_composite_maps(res[v], lab + " anomaly", vmax=vmax, masks=masks,
                                  title=f"{lab} — composite anomaly (baseline {cfg.BASELINE})")
    fig.savefig(cfg.FIG / f"06_{v}_maps.png", dpi=200, bbox_inches="tight")

`mean_int` is only defined where MHWs occurred, so its composite mixes "how many years had MHWs" with
"how hot they were"; interpret alongside `mhw_days`.

## Regional shelf-mean lag evolution (monthly MHW days)

In [ ]:
reg = cfg.regional_means(ym.mhw_days, masks)
res_reg = {k: cfg.composite_bootstrap(reg, groups[k]) for k in comps}
indiv = (cfg.detrend_years(reg) if cfg.COMPOSITE_DETREND else reg)
indiv = indiv - indiv.mean("enso_year")
indiv.attrs["years"] = groups["Strong El Niño"]
fig = cfg.plot_regional_lags(res_reg, "MHW days / month (anom.)", names, individual=indiv)
fig.savefig(cfg.FIG / "06_mhw_days_regional.png", dpi=200, bbox_inches="tight")

In [ ]:
# Fraction of shelf area in MHW, per ENSO year and season, by region (table for the paper)
frac = cfg.regional_means((seas.mhw_days > 0).where(masks.ocean).astype(float), masks)
frac_df = frac.to_dataframe(name="frac").frac.unstack(["region", "season"]).round(2)
frac_df.index = [table.label[y] for y in frac_df.index]
frac_df

In [ ]:
for v in res:
    xr.concat([r.drop_vars(["null_lo", "null_hi"], errors="ignore").expand_dims(composite=[k])
               for k, r in res[v].items()], "composite").to_netcdf(cfg.OUT / f"composites_{v}_maps.nc")
xr.concat([r.expand_dims(composite=[k]) for k, r in res_reg.items()], "composite").to_netcdf(cfg.OUT / "composites_mhw_days_regional.nc")